In [1]:
# ═══════════════════════════════════════════════════════════════════════════════
# PHASE 5 — COINTEGRATION & LEAD-LAG
# Inputs:  outputs/02_returns_matrix.parquet
#          outputs/04_correlation_spearman_full.parquet
#          outputs/03_coin_metadata.csv
# Outputs: outputs/10_cointegration_results.csv
#          outputs/11_granger_causality.csv
#          outputs/phase5_vecm_spreads.csv
#          outputs/phase5_cluster_leaders.csv
# ═══════════════════════════════════════════════════════════════════════════════

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from pathlib import Path
from itertools import combinations, permutations

from statsmodels.tsa.stattools import coint, grangercausalitytests
from statsmodels.tsa.vector_ar.vecm import coint_johansen
from statsmodels.stats.multitest import multipletests

Path("outputs").mkdir(exist_ok=True)


# ═══════════════════════════════════════════════════════════════════════════════
# LOAD INPUTS
# ═══════════════════════════════════════════════════════════════════════════════
print("Loading inputs...")

log_returns  = pd.read_parquet("outputs/02_returns_matrix.parquet")
spearman_mat = pd.read_parquet("outputs/04_correlation_spearman_full.parquet")
coin_meta    = pd.read_csv("outputs/03_coin_metadata.csv")

# Load Phase 2 cluster assignments
cluster_df   = pd.read_csv("outputs/phase2_clustering_baseline.csv")

coins = log_returns.columns.tolist()
N     = len(coins)

# Reconstruct log-prices from returns (cumsum)
log_prices = log_returns.cumsum()

print(f"  Coins         : {N}")
print(f"  Return matrix : {log_returns.shape}")
print(f"  Price matrix  : {log_prices.shape}")

# Try loading Phase 4 strong candidates if available
try:
    strong_candidates = pd.read_csv("outputs/phase4_strong_candidates.csv")
    has_phase4 = True
    print(f"  Phase 4 strong candidates : {len(strong_candidates)} pairs loaded")
except FileNotFoundError:
    strong_candidates = pd.DataFrame()
    has_phase4 = False
    print(f"  Phase 4 strong candidates : not available — using correlation filter only")


# ═══════════════════════════════════════════════════════════════════════════════
# PRE-FILTER: ONLY TEST PAIRS WITH SPEARMAN ρ > 0.60
# ═══════════════════════════════════════════════════════════════════════════════
# Cointegration is computationally expensive and theoretically only meaningful
# for pairs that already show substantial co-movement. Testing all 7,750 pairs
# would waste compute on pairs that are clearly not cointegrated.
# We pre-filter to pairs with ρ > 0.60 — roughly the top 50% of our universe.

print("\n── Pre-filtering pairs (Spearman ρ > 0.60) ────────────────────────────────")

candidate_pairs = []
for i, j in combinations(range(N), 2):
    ci, cj = coins[i], coins[j]
    rho    = spearman_mat.loc[ci, cj] if (ci in spearman_mat.index and
                                           cj in spearman_mat.columns) else np.nan
    if not np.isnan(rho) and rho > 0.60:
        candidate_pairs.append((ci, cj, rho))

candidate_df = pd.DataFrame(candidate_pairs, columns=["coin_a", "coin_b", "spearman_rho"])
print(f"  Pairs with ρ > 0.60 : {len(candidate_df):,}  (from {N*(N-1)//2:,} total)")


# ═══════════════════════════════════════════════════════════════════════════════
# 5.1  ENGLE-GRANGER COINTEGRATION TEST
# ═══════════════════════════════════════════════════════════════════════════════
# Cointegration means two price series are individually non-stationary (random
# walks) but their LINEAR COMBINATION is stationary — they share a long-run
# equilibrium and always drift back toward each other.
#
# Think of it like two dogs on a long leash. Each dog wanders randomly, but
# the leash means they can never stray too far apart. If we cut the leash
# (the cointegrating relationship breaks), they drift apart permanently.
#
# Why this matters for fund transfers: if A and B are cointegrated, holding
# either gives essentially the same long-run exposure. Transferring between
# them adds no new information.
#
# Engle-Granger: regress log_price_A on log_price_B, test residuals for
# stationarity. If residuals are stationary → cointegrated.

print("\n── Phase 5.1 │ Engle-Granger Cointegration Tests ──────────────────────────")
print(f"  Testing {len(candidate_df):,} candidate pairs...")

eg_rows = []
raw_pvals = []

for _, row in candidate_df.iterrows():
    ca, cb = row["coin_a"], row["coin_b"]

    pa = log_prices[ca].dropna()
    pb = log_prices[cb].dropna()

    # Align on common dates
    common = pa.index.intersection(pb.index)
    if len(common) < 100:
        continue

    pa_c = pa.loc[common].values
    pb_c = pb.loc[common].values

    try:
        # Engle-Granger: test both directions, take the more significant one
        stat_ab, pval_ab, _ = coint(pa_c, pb_c, autolag="AIC")
        stat_ba, pval_ba, _ = coint(pb_c, pa_c, autolag="AIC")

        # Take direction with lower p-value
        if pval_ab <= pval_ba:
            stat, pval, direction = stat_ab, pval_ab, f"{ca}→{cb}"
        else:
            stat, pval, direction = stat_ba, pval_ba, f"{cb}→{ca}"

        # Estimate spread and half-life of mean reversion
        # Spread = pa - β*pb (OLS residuals)
        beta      = np.polyfit(pb_c, pa_c, 1)[0]
        spread    = pa_c - beta * pb_c
        spread_lag= spread[:-1]
        spread_d  = np.diff(spread)

        # Half-life via AR(1) on spread: Δspread = λ*spread_{t-1} + ε
        # Half-life = -log(2) / log(1 + λ)
        if len(spread_lag) > 10:
            ar_coef   = np.polyfit(spread_lag, spread_d, 1)[0]
            if ar_coef < 0:
                half_life = -np.log(2) / np.log(1 + ar_coef)
            else:
                half_life = np.inf
        else:
            half_life = np.nan

        eg_rows.append({
            "coin_a"         : ca,
            "coin_b"         : cb,
            "eg_stat"        : round(stat, 4),
            "eg_pval_raw"    : round(pval, 6),
            "direction"      : direction,
            "beta"           : round(beta, 4),
            "spread_std"     : round(spread.std(), 6),
            "half_life_days" : round(half_life, 1) if np.isfinite(half_life) else np.nan,
            "n_obs"          : len(common),
        })
        raw_pvals.append(pval)

    except Exception:
        continue

eg_df = pd.DataFrame(eg_rows)

# ─── BH-FDR correction ────────────────────────────────────────────────────────
if len(raw_pvals) > 0:
    reject, pvals_adj, _, _ = multipletests(raw_pvals, alpha=0.05, method="fdr_bh")
    eg_df["eg_pval_adj"]     = pvals_adj
    eg_df["cointegrated"]    = reject
else:
    eg_df["eg_pval_adj"]  = np.nan
    eg_df["cointegrated"] = False

n_coint = eg_df["cointegrated"].sum()
print(f"\n  Cointegrated pairs (BH-adjusted p < 0.05) : {n_coint:,} / {len(eg_df):,}")
print(f"  Cointegration rate                        : {n_coint/len(eg_df)*100:.1f}%")

if n_coint > 0:
    # Half-life distribution for cointegrated pairs
    hl = eg_df.loc[eg_df["cointegrated"], "half_life_days"].dropna()
    print(f"\n  Mean reversion half-life (cointegrated pairs):")
    print(f"    Median : {hl.median():.1f} days")
    print(f"    Mean   : {hl.mean():.1f} days")
    print(f"    Fast (< 10 days)  : {(hl < 10).sum():,} pairs")
    print(f"    Medium (10–30d)   : {((hl >= 10) & (hl <= 30)).sum():,} pairs")
    print(f"    Slow (> 30 days)  : {(hl > 30).sum():,} pairs")

    # Top cointegrated pairs by strength (lowest adjusted p-value)
    print(f"\n  Top 20 most strongly cointegrated pairs:")
    top_coint = (eg_df[eg_df["cointegrated"]]
                 .sort_values("eg_pval_adj")
                 .head(20)[["coin_a", "coin_b", "eg_pval_adj",
                             "half_life_days", "beta"]])
    print(top_coint.to_string(index=False))


# ═══════════════════════════════════════════════════════════════════════════════
# 5.2  THRESHOLD COINTEGRATION — BAND WIDTH & REVERSION SPEED
# ═══════════════════════════════════════════════════════════════════════════════
# Standard cointegration says "the spread mean-reverts."
# Threshold cointegration (TVECM) says "it only mean-reverts OUTSIDE a band."
# Inside the band, price differences are too small to arbitrage away,
# so the spread just wanders freely.
#
# We estimate this simply: fit the spread's AR(1) separately for observations
# inside and outside a ±1σ band. If outside-band reversion is much faster
# than inside, threshold cointegration is present.
#
# For fund transfer decisions: narrow band + fast outside reversion →
# the two assets are tightly coupled. Wide band → they can diverge for
# long periods before correcting.

print("\n── Phase 5.2 │ Threshold Cointegration (Band Estimation) ──────────────────")

tvecm_rows = []
coint_pairs = eg_df[eg_df["cointegrated"]][["coin_a", "coin_b", "beta"]].values

print(f"  Estimating bands for {len(coint_pairs):,} cointegrated pairs...")

for ca, cb, beta in coint_pairs:
    pa   = log_prices[ca].dropna()
    pb   = log_prices[cb].dropna()
    common = pa.index.intersection(pb.index)
    if len(common) < 60:
        continue

    spread    = pa.loc[common].values - float(beta) * pb.loc[common].values
    spread_mu = spread.mean()
    spread_sd = spread.std()

    # Band = ±1σ around mean
    band_upper = spread_mu + spread_sd
    band_lower = spread_mu - spread_sd

    inside_mask  = (spread >= band_lower) & (spread <= band_upper)
    outside_mask = ~inside_mask

    spread_lag = spread[:-1]
    spread_d   = np.diff(spread)

    # AR(1) reversion speed inside band
    if inside_mask[:-1].sum() > 10:
        coef_in  = np.polyfit(spread_lag[inside_mask[:-1]],
                              spread_d[inside_mask[:-1]], 1)[0]
    else:
        coef_in  = np.nan

    # AR(1) reversion speed outside band
    if outside_mask[:-1].sum() > 10:
        coef_out = np.polyfit(spread_lag[outside_mask[:-1]],
                              spread_d[outside_mask[:-1]], 1)[0]
    else:
        coef_out = np.nan

    # Half-lives
    hl_in  = (-np.log(2) / np.log(1 + coef_in)
               if (not np.isnan(coef_in) and coef_in < 0) else np.nan)
    hl_out = (-np.log(2) / np.log(1 + coef_out)
               if (not np.isnan(coef_out) and coef_out < 0) else np.nan)

    tvecm_rows.append({
        "coin_a"           : ca,
        "coin_b"           : cb,
        "band_width_sigma" : round(spread_sd * 2, 6),   # full ±1σ width
        "pct_inside_band"  : round(inside_mask.mean() * 100, 1),
        "half_life_inside" : round(hl_in,  1) if hl_in  and np.isfinite(hl_in)  else np.nan,
        "half_life_outside": round(hl_out, 1) if hl_out and np.isfinite(hl_out) else np.nan,
        "threshold_effect" : (not np.isnan(hl_in) and not np.isnan(hl_out) and
                              hl_out < hl_in * 0.7),  # outside reverts 30% faster
    })

tvecm_df = pd.DataFrame(tvecm_rows)

if len(tvecm_df):
    threshold_pairs = tvecm_df["threshold_effect"].sum()
    print(f"  Pairs with clear threshold effect : {threshold_pairs} / {len(tvecm_df)}")
    print(f"  Median band width (±σ)            : {tvecm_df['band_width_sigma'].median():.4f}")
    print(f"  Median % time inside band         : {tvecm_df['pct_inside_band'].median():.1f}%")

    # Tightest bands (strongest coupling)
    print(f"\n  Top 15 tightest-band cointegrated pairs:")
    print(tvecm_df.sort_values("band_width_sigma")
          .head(15)[["coin_a", "coin_b", "band_width_sigma",
                      "half_life_outside", "threshold_effect"]]
          .to_string(index=False))


# ═══════════════════════════════════════════════════════════════════════════════
# 5.3  GRANGER CAUSALITY — LEAD-LAG WITHIN CLUSTERS
# ═══════════════════════════════════════════════════════════════════════════════
# Granger causality asks: does knowing coin A's past returns help predict
# coin B's future returns — BEYOND what B's own past already tells us?
# If yes, A "Granger-causes" B → A is the LEADER, B is the FOLLOWER.
#
# Within a cluster, we always want to hold the leader:
#   - It moves first, others follow
#   - Selling the leader and buying the follower gives you lagged exposure
#     to the same signal, with extra transaction costs
#
# We run this within each Phase 2 cluster, and also within the cointegrated
# pairs group.

print("\n── Phase 5.3 │ Granger Causality — Leader Detection ───────────────────────")

MAX_LAG     = 5    # test up to 5-day lead-lag
GRANGER_MIN_OBS = 100

def run_granger_pair(series_a: pd.Series,
                     series_b: pd.Series,
                     max_lag: int = MAX_LAG) -> dict:
    """
    Tests A→B and B→A Granger causality.
    Returns p-values for both directions at each lag.
    """
    common = series_a.dropna().index.intersection(series_b.dropna().index)
    if len(common) < GRANGER_MIN_OBS:
        return None

    xa = series_a.loc[common].values
    xb = series_b.loc[common].values

    data_ab = np.column_stack([xb, xa])  # [caused, cause]
    data_ba = np.column_stack([xa, xb])

    try:
        res_ab = grangercausalitytests(data_ab, maxlag=max_lag, verbose=False)
        res_ba = grangercausalitytests(data_ba, maxlag=max_lag, verbose=False)

        # Take minimum p-value across lags (most significant lag)
        pval_ab = min(res_ab[lag][0]["ssr_ftest"][1] for lag in range(1, max_lag + 1))
        pval_ba = min(res_ba[lag][0]["ssr_ftest"][1] for lag in range(1, max_lag + 1))

        return {"pval_ab": pval_ab, "pval_ba": pval_ba}
    except Exception:
        return None


# ─── Run Granger within each Phase 2 cluster ─────────────────────────────────
granger_rows = []
all_pvals    = []

cluster_ids = cluster_df["cluster_id"].unique()
print(f"\n  Running Granger causality within {len(cluster_ids)} clusters...")

for cid in sorted(cluster_ids):
    members = cluster_df.loc[cluster_df["cluster_id"] == cid, "coin_id"].tolist()
    members = [m for m in members if m in coins]

    if len(members) < 2:
        continue

    for ca, cb in permutations(members, 2):
        result = run_granger_pair(log_returns[ca], log_returns[cb])
        if result is None:
            continue

        granger_rows.append({
            "cluster_id"   : cid,
            "coin_a"       : ca,
            "coin_b"       : cb,
            "direction"    : f"{ca}→{cb}",
            "pval_raw"     : result["pval_ab"],
        })
        all_pvals.append(result["pval_ab"])

# Also run Granger on cointegrated pairs not already covered
if n_coint > 0:
    coint_set = set(zip(eg_df[eg_df["cointegrated"]]["coin_a"],
                        eg_df[eg_df["cointegrated"]]["coin_b"]))
    existing  = set(zip([r["coin_a"] for r in granger_rows],
                        [r["coin_b"] for r in granger_rows]))

    extra_pairs = [(ca, cb) for ca, cb in coint_set if (ca, cb) not in existing]
    print(f"  Running Granger on {len(extra_pairs)} additional cointegrated pairs...")

    for ca, cb in extra_pairs:
        for a, b in [(ca, cb), (cb, ca)]:
            result = run_granger_pair(log_returns[a], log_returns[b])
            if result is None:
                continue
            granger_rows.append({
                "cluster_id": -1,   # cointegrated group
                "coin_a"    : a,
                "coin_b"    : b,
                "direction" : f"{a}→{b}",
                "pval_raw"  : result["pval_ab"],
            })
            all_pvals.append(result["pval_ab"])

granger_df = pd.DataFrame(granger_rows)

# ─── BH correction on all Granger p-values ───────────────────────────────────
if len(all_pvals) > 0:
    reject, pvals_adj, _, _ = multipletests(all_pvals, alpha=0.05, method="fdr_bh")
    granger_df["pval_adj"]   = pvals_adj
    granger_df["significant"]= reject
else:
    granger_df["pval_adj"]   = np.nan
    granger_df["significant"]= False

n_sig_granger = granger_df["significant"].sum()
print(f"\n  Significant Granger causal directions (BH-adjusted) : {n_sig_granger}")


# ─── Identify leader per cluster ─────────────────────────────────────────────
# Leader = coin that significantly Granger-causes the most others in the cluster
# while being caused by the fewest.
# Tiebreaker: highest median daily volume.

print("\n  Identifying cluster leaders...")

leader_rows = []

for cid in sorted(cluster_ids):
    members   = cluster_df.loc[cluster_df["cluster_id"] == cid, "coin_id"].tolist()
    members   = [m for m in members if m in coins]
    if len(members) < 2:
        leader_rows.append({
            "cluster_id": cid, "leader": members[0] if members else "unknown",
            "causes_count": 0, "caused_by_count": 0, "leader_reason": "only member"
        })
        continue

    sig_dir = granger_df[
        (granger_df["cluster_id"] == cid) & (granger_df["significant"] == True)
    ]

    # Count: how many coins does each member Granger-cause?
    causes_count    = sig_dir.groupby("coin_a")["coin_b"].count().reindex(members, fill_value=0)
    # Count: how many coins Granger-cause each member?
    caused_by_count = sig_dir.groupby("coin_b")["coin_a"].count().reindex(members, fill_value=0)

    # Leadership score = causes - caused_by
    leadership_score = causes_count - caused_by_count

    if leadership_score.max() == 0:
        # No clear Granger leader — use most liquid coin
        vols = coin_meta.set_index("coin_id")["median_daily_vol"].reindex(members)
        leader = vols.idxmax() if not vols.isna().all() else members[0]
        reason = "most liquid (no Granger signal)"
    else:
        # Get candidates with highest leadership score
        top_score = leadership_score.max()
        candidates = leadership_score[leadership_score == top_score].index.tolist()
        if len(candidates) == 1:
            leader = candidates[0]
            reason = f"Granger-causes {int(causes_count[leader])} others"
        else:
            # Tiebreak by liquidity
            vols   = coin_meta.set_index("coin_id")["median_daily_vol"].reindex(candidates)
            leader = vols.idxmax() if not vols.isna().all() else candidates[0]
            reason = f"Granger tied — most liquid tiebreak"

    leader_rows.append({
        "cluster_id"      : cid,
        "leader"          : leader,
        "n_members"       : len(members),
        "members"         : ";".join(members),
        "causes_count"    : int(causes_count.get(leader, 0)),
        "caused_by_count" : int(caused_by_count.get(leader, 0)),
        "leader_reason"   : reason,
    })

leader_df = pd.DataFrame(leader_rows)

print(f"\n  Cluster leaders:")
for _, row in leader_df.iterrows():
    print(f"    Cluster {row['cluster_id']:2d} │ Leader: {row['leader']:25s} │ "
          f"{row['n_members']} members │ {row['leader_reason']}")


# ═══════════════════════════════════════════════════════════════════════════════
# 5.4  MERGE COINTEGRATION + TVECM + GRANGER INTO FINAL PAIRS TABLE
# ═══════════════════════════════════════════════════════════════════════════════
print("\n── Phase 5.4 │ Building Final Cointegration Summary ───────────────────────")

# Merge EG results with TVECM band estimates
final_coint = eg_df.merge(
    tvecm_df[["coin_a", "coin_b", "band_width_sigma",
              "half_life_outside", "threshold_effect",
              "pct_inside_band"]],
    on=["coin_a", "coin_b"], how="left"
)

# Add Phase 4 cross-reference if available
if has_phase4 and len(strong_candidates):
    sc_flag = strong_candidates[["coin_a", "coin_b"]].copy()
    sc_flag["phase4_confirmed"] = True
    final_coint = final_coint.merge(sc_flag, on=["coin_a", "coin_b"], how="left")
    final_coint["phase4_confirmed"] = final_coint["phase4_confirmed"].fillna(False)
else:
    final_coint["phase4_confirmed"] = False

# Redundancy strength for cointegrated pairs
# Strong = cointegrated + fast reversion + narrow band
def coint_strength(row) -> str:
    if not row["cointegrated"]:
        return "NOT_COINTEGRATED"
    hl = row.get("half_life_days", np.nan)
    bw = row.get("band_width_sigma", np.nan)
    if pd.isna(hl):
        return "COINTEGRATED_UNKNOWN_SPEED"
    if hl < 10 and (pd.isna(bw) or bw < 0.05):
        return "STRONG"     # fast reversion, narrow band
    elif hl < 30:
        return "MODERATE"   # medium reversion
    else:
        return "WEAK"       # cointegrated but slow reversion

final_coint["coint_strength"] = final_coint.apply(coint_strength, axis=1)

strength_counts = final_coint["coint_strength"].value_counts()
print(f"\n  Cointegration strength distribution:")
for s, cnt in strength_counts.items():
    print(f"    {s:30s} : {cnt:,}")

print(f"\n  Top 20 strongest cointegrated pairs:")
strong_coint = (final_coint[final_coint["cointegrated"]]
                .sort_values(["coint_strength", "eg_pval_adj"])
                .head(20)[["coin_a", "coin_b", "eg_pval_adj",
                            "half_life_days", "coint_strength",
                            "band_width_sigma"]])
print(strong_coint.to_string(index=False))


# ═══════════════════════════════════════════════════════════════════════════════
# SAVE OUTPUTS
# ═══════════════════════════════════════════════════════════════════════════════
print("\n── Saving Phase 5 outputs ──────────────────────────────────────────────────")

final_coint.to_csv("outputs/10_cointegration_results.csv", index=False)
print("  ✅ 10_cointegration_results.csv     → EG test + TVECM band + strength")

granger_df.to_csv("outputs/11_granger_causality.csv", index=False)
print("  ✅ 11_granger_causality.csv         → Granger directions + BH p-values")

leader_df.to_csv("outputs/phase5_cluster_leaders.csv", index=False)
print("  ✅ phase5_cluster_leaders.csv       → leader coin per cluster")

tvecm_df.to_csv("outputs/phase5_vecm_spreads.csv", index=False)
print("  ✅ phase5_vecm_spreads.csv          → band width + reversion speed")

print(f"""
╔══════════════════════════════════════════════════════════════════╗
║  PHASE 5 COMPLETE                                                ║
╠══════════════════════════════════════════════════════════════════╣
║  Candidate pairs tested     : {len(eg_df):,}                          ║
║  Cointegrated pairs         : {n_coint:,}                            ║
║  Cointegration rate         : {n_coint/max(len(eg_df),1)*100:.1f}%                         ║
║  Significant Granger dirs   : {n_sig_granger:,}                          ║
║  Cluster leaders identified : {len(leader_df)}                             ║
╠══════════════════════════════════════════════════════════════════╣
║  Next: Phase 6 — Network Construction                           ║
║  Key inputs for Phase 6:                                         ║
║    02_returns_matrix.parquet                                     ║
║    06_tail_dependence_lower.parquet  ← λL for MST               ║
║    10_cointegration_results.csv      ← edge weights             ║
╚══════════════════════════════════════════════════════════════════╝
""")

Loading inputs...
  Coins         : 125
  Return matrix : (729, 125)
  Price matrix  : (729, 125)
  Phase 4 strong candidates : 3613 pairs loaded

── Pre-filtering pairs (Spearman ρ > 0.60) ────────────────────────────────
  Pairs with ρ > 0.60 : 6,178  (from 7,750 total)

── Phase 5.1 │ Engle-Granger Cointegration Tests ──────────────────────────
  Testing 6,178 candidate pairs...

  Cointegrated pairs (BH-adjusted p < 0.05) : 43 / 6,178
  Cointegration rate                        : 0.7%

  Mean reversion half-life (cointegrated pairs):
    Median : 15.0 days
    Mean   : 41.4 days
    Fast (< 10 days)  : 11 pairs
    Medium (10–30d)   : 30 pairs
    Slow (> 30 days)  : 2 pairs

  Top 20 most strongly cointegrated pairs:
                   coin_a              coin_b  eg_pval_adj  half_life_days   beta
                 algorand peanut-the-squirrel     0.000215            19.4 0.4221
              the-sandbox             vechain     0.000544             5.2 1.0594
                     b